In [2]:
import torch
import sys, os

# 确保能 import model 目录
sys.path.insert(0, os.path.abspath('.'))

print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.mps.is_available else 'cpu'
print(f'\n使用设备: {device}')

PyTorch 版本: 2.6.0
CUDA 可用: False

使用设备: mps


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('./model')
print(f'Tokenizer 加载成功，词表大小: {tokenizer.vocab_size}')
print(f'bos_token_id: {tokenizer.bos_token_id}, eos_token_id: {tokenizer.eos_token_id}')

Tokenizer 加载成功，词表大小: 6400
bos_token_id: 1, eos_token_id: 2


In [3]:
from model.model_minimind import MiniMindConfig

from model.model_minimind import MiniMindForCausalLM

WEIGHT_PATH = './out/full_sft_768.pth'

config = MiniMindConfig(
    hidden_size=768,
    num_hidden_layers=16,    
    num_attention_heads=8,
    num_key_value_heads=2,
    vocab_size=6400,
    use_moe=False,    
)

model = MiniMindForCausalLM(config)

# 加载 .pth state dict
state_dict = torch.load(WEIGHT_PATH, map_location=device)

# 兼容带/不带 'model.' 前缀的权重
if all(k.startswith('model.') for k in state_dict.keys()):
    state_dict = {k[len('model.'):]: v for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'Missing keys  : {missing}')
print(f'Unexpected keys: {unexpected}')

model = model.to(device).eval()

total = sum(p.numel() for p in model.parameters())
print(f'\n模型加载成功 ✓  参数量: {total/1e6:.1f}M')

print(f'已加载权重: {WEIGHT_PATH}')

/Users/kevinpan/.pyenv/versions/LLM/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Missing keys  : ['model.layers.0.mlp.up_proj.bias', 'model.layers.1.mlp.up_proj.bias', 'model.layers.2.mlp.up_proj.bias', 'model.layers.3.mlp.up_proj.bias', 'model.layers.4.mlp.up_proj.bias', 'model.layers.5.mlp.up_proj.bias', 'model.layers.6.mlp.up_proj.bias', 'model.layers.7.mlp.up_proj.bias', 'model.layers.8.mlp.up_proj.bias', 'model.layers.9.mlp.up_proj.bias', 'model.layers.10.mlp.up_proj.bias', 'model.layers.11.mlp.up_proj.bias', 'model.layers.12.mlp.up_proj.bias', 'model.layers.13.mlp.up_proj.bias', 'model.layers.14.mlp.up_proj.bias', 'model.layers.15.mlp.up_proj.bias']
Unexpected keys: []

模型加载成功 ✓  参数量: 104.1M
已加载权重: ./out/full_sft_768.pth


In [4]:
def generate(
    prompt: str,
    mode: str = 'sft',          
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
):
    """
    mode='sft'     : 使用 chat template，适合 full_sft 权重
    mode='pretrain': 直接续写 prompt，适合 pretrain 权重
    """
    if mode == 'sft':
        messages = [{'role': 'user', 'content': prompt}]
        input_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt'
        ).to(device)
    else:
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        output_ids = input_ids.clone()
        past_key_values = None

        for _ in range(max_new_tokens):
            outputs = model(
                input_ids=input_ids if past_key_values is None else input_ids[:, -1:],
                past_key_values=past_key_values,
                use_cache=True
            )
            logits = outputs.logits[:, -1, :]  # [1, vocab_size]
            past_key_values = outputs.past_key_values

            # Repetition penalty
            if repetition_penalty != 1.0:
                for token_id in set(output_ids[0].tolist()):
                    logits[0, token_id] /= repetition_penalty

            # Temperature + Top-p sampling
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cumulative = torch.cumsum(sorted_probs, dim=-1)
            sorted_probs[cumulative - sorted_probs > top_p] = 0
            sorted_probs /= sorted_probs.sum()
            next_token = sorted_idx[0, torch.multinomial(sorted_probs[0], 1)]

            if next_token.item() == tokenizer.eos_token_id:
                break

            output_ids = torch.cat([output_ids, next_token.unsqueeze(0).unsqueeze(0)], dim=1)
            input_ids = next_token.unsqueeze(0).unsqueeze(0)

    # 只解码新生成的部分
    new_tokens = output_ids[0, input_ids.shape[1] if mode == 'pretrain' else 
                            tokenizer.apply_chat_template([{'role':'user','content':prompt}], 
                            add_generation_prompt=True, return_tensors='pt').shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


# 更简洁的包装（使用 model.generate）
def chat(prompt: str, max_new_tokens: int = 256, temperature: float = 0.7):
    """直接使用 HuggingFace generate 接口（推荐）"""
    messages = [{'role': 'user', 'content': prompt}]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        output[0][input_ids.shape[1]:], skip_special_tokens=True
    )
    return response


print('推理函数已定义 ✓')
print('  chat(prompt)      → SFT 指令对话')
print('  generate(prompt, mode="pretrain") → 续写')

推理函数已定义 ✓
  chat(prompt)      → SFT 指令对话
  generate(prompt, mode="pretrain") → 续写


In [10]:
# README 对比表格所需的 4 组 prompt
test_prompts = [
    '1 + 1 = ?',
    '介绍一下你自己',
    '中国的首都是哪里？',
    '用一句话解释机器学习',
    '写一首关于春天的诗',
]

print('=' * 60)
print('SFT 模型输出（用于填写 README 对比表格）')
print('=' * 60)

for prompt in test_prompts:
    print(f'\n【Prompt】{prompt}')
    response = chat(prompt, max_new_tokens=150)
    print(f'【输出】{response}')
    print('-' * 40)

SFT 模型输出（用于填写 README 对比表格）

【Prompt】1 + 1 = ?
【输出】这是一个简单的Python程序，可以生成一个包含10个数字的列表。 下面是实现：```python# 定义要对列表中的所有数字的值numbers = []# 将列表中的所有元素相加numbers.sort()# 打印输出列表print(numbers)```该程序首先定义了一个列表`numbers`，然后使用`sort()`函数将列表中的所有元素相加，并将其存储在变量`numbers`中。最后，我们使用`print()`函数打印输出结果。当你运行此程序时，输出应该是`[1, 2, 3, 4, 5]`。
----------------------------------------

【Prompt】介绍一下你自己
【输出】作为一个人工智能语言模型，我没有感情和偏好。但我可以给您提供一些关于个人成长的详细信息，供您参考：1. 个人兴趣爱好：从不同领域的角度看，个人兴趣爱好包括兴趣爱好、兴趣爱好、职业目标等等。2. 自我成长：了解自己的优点、缺点、兴趣爱好等方面，并通过这些方面进行成长。3. 学习能力：学习者需要不断地学习和探索，并保持积极的态度，以应对未来的挑战和机遇。4. 社交技能：与人交往是个人发展的关键，与他人建立良好关系可以提高个人形象和自我价值感。5. 自我
----------------------------------------

【Prompt】中国的首都是哪里？
【输出】中国的首都是华语城。华语城是中国古代的历史文化名城，位于中国东部沿海市。华语城是中国古代的军事建筑之一，也是世界上文化遗产。华语城是中国古代皇帝、商人和皇家宫殿等民族建筑。华语城是中国古代文化的重要中心，也是世界文化遗产。华语城是中国古代皇家文化的代表，被誉为“世界文化之都”，有着丰富的历史和文化价值。华语城是中国古代文明的重要组成部分，也被称为中国古代皇帝的宗旨。华语城是一个充满
----------------------------------------

【Prompt】用一句话解释机器学习
【输出】机器学习是一种人工智能的分支，通过让计算机从数据中学习模式和规律来改进其性能。
----------------------------------------

【Pr

In [ ]:
def load_weights(weight_path: str):
    """动态切换权重"""
    state_dict = torch.load(weight_path, map_location=device)
    if all(k.startswith('model.') for k in state_dict.keys()):
        state_dict = {k[len('model.'):]: v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=False)
    print(f'已切换权重: {weight_path}')


def pretrain_generate(prompt: str, max_new_tokens: int = 100):
    """Pretrain 续写模式"""
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=1.0,
            do_sample=True,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)


test_prompt = '介绍一下你自己'

# Pretrain 输出
load_weights('./out/pretrain_768.pth')
pt_output = pretrain_generate(test_prompt)
print(f'[Pretrain] {test_prompt}')
print(f'输出: {pt_output[:200]}')

print()

# SFT 输出
load_weights('./out/full_sft_768.pth')
sft_output = chat(test_prompt)
print(f'[SFT] {test_prompt}')
print(f'输出: {sft_output}')

已切换权重: ./out/pretrain_768.pth
[Pretrain] 介绍一下你自己
输出: 天下：听听�粉是都碄�至头一，听是不意口骨灷的像鸾，道

已切换权重: ./out/full_sft_768.pth
[SFT] 介绍一下你自己
输出: 作为一个AI，我没有自己的能力和智力，无法真正理解人类的本质和意义。但是我可以给你提供一些关于人类的有趣事实和数据。1. 人类的进化是一个非常复杂的概念，人类的进化是自然界中最复杂的个体，包括了生物、植物、动物等各种物种。这些物种之间的互动是通过人类的遗传学和认知行为来获取的。2. 人与人之间存在着相互依存的关系。人与人之间有着密切的联系，这些关系可以被看作是人类自我实现的结果。在人类社会中，许多人会选择从不同角度去思考和决策，而忽略了不同地理位置和文化背景。3. 人类的历史可以追溯到数千年前的古代文明时期。这个时期的历史可以追溯到公元前6世纪，但现在已经成为文明发展的中心。4. 在人类历史上，人们开始探索人类的起源和发展。随着时间的推移，人类开始逐渐认识到自己的起源和发展，并在过去的发展中不断探索和发展。5. 人类的生存需要时间和资源。


In [12]:
your_question = '请介绍一下 Transformer 架构的核心组件'

response = chat(your_question, max_new_tokens=300, temperature=0.7)
print(f'Q: {your_question}')
print(f'A: {response}')

Q: 请介绍一下 Transformer 架构的核心组件
A: Sweet School 架构的核心组件是 WHERL ，它由两个核心组件组成，每个核心组件包含一个或多个核心组件。下面是一些基本的 DIFO COUNT 核心组件：
1. THERL （在 ROC 系统中）和 UPDATA（在 VS OV- APIC 系统中）组成的集合。
2. THERL （在 HTML 系统中）和 PHERL （在 ROC 系统中）之间创建的 PHERL 核心组件。
3.  THERL （在 VS OV- ORDER BY）和 DIFO 中组成的集合。
4. THERL （在 ORDER BY 系统中）和 PHERL （在 ROC 系统中）之间创建的 PHERL 核心组件。
5. THERL （在 ROC 系统中）和 LOC 中创建的 PHERL 核心组件。
6. THERL （在 HTER C
